## Importing relevant libraries

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
import time

## Target URL for scraping

In [ ]:
url="https://en.wikipedia.org/w/index.php?title=Category:2025_films&pagefrom=Sound+of+Falling#mw-pages"
source=requests.get(url)
soup=BeautifulSoup(source.text,'html')

## Extracting links to all the individual pages of the movies

In [ ]:
category_div = soup.find_all("div", class_="mw-content-ltr")[2]
a_tags=category_div.find_all('a')
links=[]
for a in a_tags:
  link=a['href']
  links.append(link)

## Setting up pandas DataFrame to store collected data

In [ ]:
cols = [
    "title",
    "description",
    "directed_by",
    "written_by",
    "produced_by",
    "starring",
    "cinematography",
    "edited_by",
    "release_date",
    "country",
    "language"
]

df=pd.DataFrame(columns=cols)


## Looping code with extraction logic as well as ploite use to prevent getting blocked

In [ ]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36'
}

def get_title(soup):
    title_tag = soup.find('i')
    return title_tag.text.strip() if title_tag else None

def get_description(soup):
    paragraphs = soup.find_all('p')
    if len(paragraphs) > 1:
        return paragraphs[1].text.strip()
    elif paragraphs:
        return paragraphs[0].text.strip()
    return None

for x in links:
    url2 = f'https://en.wikipedia.org{x}'
    try:
        response = requests.get(url2, headers=headers)
        if response.status_code != 200:
            print(f"Failed to fetch {url2}")
            continue

        soup2 = BeautifulSoup(response.text, 'html.parser')

        info = {
            "title": get_title(soup2),
            "description": get_description(soup2),
            "directed_by": None,
            "written_by": None,
            "produced_by": None,
            "starring": None,
            "cinematography": None,
            "edited_by": None,
            "release_date": None,
            "country": None,
            "language": None
        }

        details = soup2.find('table', class_="infobox vevent")
        if details:
            column_data = details.find_all('tr')[1:]
            for row in column_data:
                header = row.find('th')
                row_data = row.find('td')
                if header and row_data:
                    label = header.text.strip().replace('\n', ' ').lower()
                    value = row_data.get_text(separator=", ", strip=True).replace('\xa0', ' ').replace('[1]', '').replace('[citation needed]', '')

                    if "directed by" in label:
                        info["directed_by"] = value
                    elif "written by" in label:
                        info["written_by"] = value
                    elif "produced by" in label:
                        info["produced_by"] = value
                    elif "starring" in label:
                        info["starring"] = value
                    elif "cinematography" in label:
                        info["cinematography"] = value
                    elif "edited by" in label:
                        info["edited_by"] = value
                    elif "release date" in label:
                        list_items = row_data.find_all('li')
                        if list_items:
                            first_li = list_items[0]
                            for span in first_li.find_all('span'):
                                span.decompose()
                            value = first_li.get_text(strip=True).replace('\xa0', ' ')
                        else:
                            value = row_data.get_text(separator=", ", strip=True).replace('\xa0', ' ')
                        info["release_date"] = value
                    elif "country" in label or 'countries' in label:
                        info["country"] = value
                    elif "language" in label:
                        info["language"] = value

        new_df = pd.DataFrame([info])
        df = pd.concat([df, new_df], ignore_index=True)

        print(f"Scraped: {info['title']}")
        time.sleep(1.5)  # polite delay

    except Exception as e:
        print(f"Error while scraping {url2}: {e}")

Scraped: Sound of Falling
Scraped: Sovereign
Scraped: Space Cadet
Scraped: Speak
Scraped: Spinal Tap II: The End Continues
Scraped: Splitsville
Scraped: The SpongeBob Movie: Search for SquarePants
Scraped: SpongeBob: Order Up
Scraped: Sri Sri Sri Raja Vaaru
Scraped: Star Trek: Section 31
Scraped: Starwalker
Scraped: The Stepmother's Bond
Scraped: Stolen
Scraped: Stone of Destiny
Scraped: Stop the Insanity: Finding Susan Powter
Scraped: The Story of Us
Scraped: Strange Journey: The Story of Rocky Horror
Scraped: The Strangers – Chapter 2
Scraped: Straw
Scraped: Streaming
Scraped: The Stringer
Scraped: Subham
Scraped: Sukkwan Island
Scraped: Suky
Scraped: Sumathi Valavu
Scraped: Summer of 69
Scraped: Sumo
Scraped: Sunfish (& Other Stories on Green Lake)
Scraped: Sunita
Scraped: Superman
Scraped: Surviving Ohio State
Scraped: Suryapet Junction
Scraped: SuSheela SuJeet
Scraped: Sweet Dreams
Scraped: Sweet Summer Pow Wow
Scraped: Sweetheart!
Scraped: Sweetness
Scraped: Swiped
Scraped: Taand